# AI Application Architecture — Single Agent

Single Agent architecture is the **foundation of Agentic AI**. Before learning Multi-Agent systems, you must clearly understand when a **Single Agent** is sufficient.

---

# 1. What is a Single Agent?

## Definition

A **Single Agent** is one AI agent responsible for understanding the user's request, reasoning, using tools if needed, and generating the final response.

Unlike Multi-Agent systems, **all decision-making is handled by one agent**.

---

## Interview Answer

> "A Single Agent architecture consists of one intelligent agent that performs planning, reasoning, tool selection, retrieval, and response generation. It is suitable for applications where a single workflow can efficiently handle the user's request."

---

# 2. Enterprise Architecture (AWS + Azure)

```text
                                    User
                                      │
                                      ▼
             Route53 + CloudFront / Azure Front Door
                                      │
                                      ▼
       Amazon API Gateway / Azure API Management
                                      │
                                      ▼
      AWS ALB / Azure Application Gateway
                                      │
                                      ▼
    ECS Fargate / Azure Container Apps (FastAPI)
                                      │
                                      ▼
                           JWT Authentication
                                      │
                                      ▼
                            LangGraph Agent
                                      │
             ┌────────────────────────┼────────────────────────┐
             ▼                        ▼                        ▼
        Retriever               Tool Calling              Memory
             │                        │                        │
             ▼                        ▼                        ▼
 Qdrant/OpenSearch          SQL/API/Python Tool         Redis
             │
             ▼
     AWS Bedrock / Azure OpenAI
             │
             ▼
      PostgreSQL + Redis
             │
             ▼
CloudWatch + LangSmith
             │
             ▼
          Response
```

---

# 3. Internal Working

```text
User Question

↓

Agent receives question

↓

Understand Intent

↓

Need RAG?

↓

Retrieve Context

↓

Need Tool?

↓

Execute Tool

↓

Call LLM

↓

Generate Answer

↓

Return Response
```

Everything is controlled by **one agent**.

---

# 4. Responsibilities of Single Agent

The agent is responsible for:

- Understanding user intent
- Planning
- Calling tools
- Retrieving documents
- Calling the LLM
- Generating the answer
- Maintaining conversation state

---

# 5. Production Flow

Example Question

> "What is my remaining leave balance?"

```text
User

↓

FastAPI

↓

JWT Validation

↓

Redis Cache

↓

LangGraph Agent

↓

Tool Required?

↓

Payroll API

↓

Current Leave Balance

↓

Bedrock/OpenAI

↓

Natural Language Response

↓

Save Chat

↓

Return
```

---

# 6. LangGraph Representation

```text
START

↓

Agent Node

↓

Tool Needed?

├── Yes → Tool Node
│          │
│          ▼
│      Back to Agent
│
└── No

↓

Generate Answer

↓

END
```

---

# 7. Simple LangGraph Code

```python
from langgraph.graph import StateGraph

workflow = StateGraph(dict)

workflow.add_node("agent", agent_node)
workflow.add_node("tool", tool_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    "agent",
    should_call_tool,
    {
        "tool": "tool",
        "end": "__end__"
    }
)

workflow.add_edge("tool", "agent")

graph = workflow.compile()
```

---

# 8. Example Use Cases

### HR Assistant

- Leave balance
- Policies
- Holidays

One agent is enough.

---

### Payroll Assistant

- Salary slips
- Tax details
- Payroll FAQs

One agent.

---

### Healthcare FAQ

- Insurance Policy
- Coverage
- Hospital Search

One agent.

---

### Document Chatbot

- PDF Upload
- Question Answering

One agent.

---

# 9. Tools Used by Single Agent

The agent can call:

- SQL Database
- REST APIs
- Python Functions
- Search APIs
- Calculator
- Vector Database
- OCR Service

Example

```text
Agent

↓

SQL

↓

Employee Table
```

---

# 10. Memory

Single Agent can use

Short-term

↓

Redis

Long-term

↓

PostgreSQL

Conversation memory improves follow-up questions.

---

# 11. Best Practices

✅ Keep responsibilities focused.

✅ Use tools instead of putting everything into prompts.

✅ Add memory only when needed.

✅ Validate tool outputs before sending to the LLM.

✅ Cache frequent responses.

---

# 12. When to Use Single Agent?

Use when

- One workflow
- Limited tools
- Simple business process
- No specialist reasoning required
- Small-to-medium enterprise applications

---

# 13. When NOT to Use?

Don't use Single Agent when

- Multiple specialized domains exist
- Independent reasoning is required
- Complex planning
- Multiple departments
- Long workflows
- Multiple approvals

Then

↓

Multi-Agent

---

# 14. Advantages

✅ Simple

✅ Easy to maintain

✅ Lower latency

✅ Lower cost

✅ Easier debugging

---

# 15. Disadvantages

❌ One agent does everything

❌ Difficult to scale reasoning

❌ Prompt becomes large

❌ Harder to specialize

---

# 16. Single Agent vs Traditional RAG

| Traditional RAG | Single Agent |
|-----------------|--------------|
| Fixed workflow | Can reason |
| Retrieve → LLM | Decide whether retrieval is needed |
| No tools | Can call tools |
| Static | Dynamic |

---

# 17. Single Agent vs Multi-Agent

| Single Agent | Multi-Agent |
|--------------|-------------|
| One AI | Multiple AIs |
| One prompt | Specialized prompts |
| Simple workflow | Complex workflow |
| Easier | More flexible |
| Lower cost | Higher cost |

---

# 18. Common Interview Questions

### Q1. What is a Single Agent?

One AI agent that manages the complete workflow from user request to response.

---

### Q2. Can a Single Agent call multiple tools?

**Yes.**

Example:

```text
Agent

↓

SQL

↓

Weather API

↓

Calculator

↓

LLM
```

---

### Q3. Can Single Agent perform RAG?

Yes.

Retriever

↓

Vector DB

↓

LLM

↓

Answer

---

### Q4. Can it use memory?

Yes.

Redis

↓

Short-term

PostgreSQL

↓

Long-term

---

### Q5. When would you prefer a Single Agent?

When the workflow is straightforward and one agent can efficiently manage planning, retrieval, and tool execution without requiring specialized reasoning.

---

# 19. Scenario-Based Question

### Interviewer

> Design an HR AI Assistant.

Expected Answer

A **Single Agent** is sufficient because one agent can:

- Retrieve HR policies from Qdrant/OpenSearch.
- Call the HR API for leave balances.
- Use Bedrock/OpenAI to generate a natural language response.
- Store chat history in PostgreSQL.
- Cache common questions in Redis.

There's no need for multiple specialized agents.

---

# 20. EPAM Senior Answer (2–3 Minutes)

> "I choose a Single Agent architecture when a single workflow can handle the complete business process. The agent is responsible for intent understanding, planning, tool selection, retrieval, and response generation. In production, the request reaches FastAPI, where authentication and authorization are validated. The application checks Redis for cached responses before invoking a LangGraph-based agent. The agent determines whether it needs to query a vector database, call an enterprise API, or invoke another tool. After collecting the required information, it sends the context to AWS Bedrock or Azure OpenAI to generate the final response. Chat history is stored in PostgreSQL, frequently accessed responses are cached in Redis, and execution traces are captured in LangSmith while infrastructure metrics are monitored using CloudWatch or Azure Monitor. A Single Agent architecture is ideal for HR assistants, document Q&A, customer support bots, and similar use cases where one intelligent agent can manage the end-to-end workflow efficiently."